# WideBind Colab Training

**Model**: D=4096, 32 layers, 32 experts, SwiGLU, QK-RMSNorm, RoPE θ=1e6, pos_id binding
**VRAM target**: T4 (16GB) or better
**Data**: token_stream_*.bin files in Google Drive

---

In [ ]:
# @title 1. Mount Drive & Install Deps
import os, sys, math, time, glob, json

from google.colab import drive
drive.mount('/content/drive')

# Config — point this to your Drive folder with widebind/ and data/
DRIVE_ROOT = '/content/drive/MyDrive/widebind'  # @param {type:'string'}
DATA_DIR    = os.path.join(DRIVE_ROOT, 'data')
SAVE_DIR    = os.path.join(DRIVE_ROOT, 'checkpoints')
LOG_DIR     = os.path.join(DRIVE_ROOT, 'logs')

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f'DRIVE_ROOT={DRIVE_ROOT}')
print(f'DATA_DIR={DATA_DIR}')
print(f'SAVE_DIR={SAVE_DIR}')

In [ ]:
# @title 2. Clone Code from GitHub (data on Drive)
import subprocess

DST = '/content/widebind'
if not os.path.exists(DST):
    print('Cloning from GitHub...')
    subprocess.run(['git', 'clone', 'https://github.com/BlackCatSpb/widebind.git', DST], check=True)
    print('Done.')
else:
    print('Already cloned, pulling latest...')
    subprocess.run(['git', '-C', DST, 'pull'], check=True)

sys.path.insert(0, DST)
os.chdir(DST)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# @title 3. Verify GPU & Imports
import torch
import torch.nn.functional as F
import numpy as np
from torch.serialization import add_safe_globals
from core import WideBindConfig, WideBindStack, MirrorLRScheduler

add_safe_globals([WideBindConfig])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'N/A'
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if device == 'cuda' else 0
print(f'Device: {device}  GPU: {gpu_name}  VRAM: {gpu_mem:.1f} GB')
print(f'PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}')

In [ ]:
# @title 4. Build Model (T4-optimised: D=2560, 24 layers)
# Head: 'partitioned' (softmax, по умолчанию) | 'sigmoid_coded' (новая factored-голова без softmax)
HEAD_MODE = 'partitioned'  # @param ['partitioned','sigmoid_coded']

cfg = WideBindConfig(
    D=2560,
    vocab=65536,
    n_layers=24,
    bind_K=64,
    mlp_groups=32,
    mlp_expand=4,
    seq_len=512,
    lr=3e-4,
    max_steps=300000,
    warmup_steps=2000,
    log_interval=100,
    eval_interval=1000,
    save_interval=5000,
    scheduler='mirror',
    private_mem=True,
    expert_asymmetry=True,
    meta_trust=True,
    data_dir=DATA_DIR,
    save_dir=SAVE_DIR,
    log_dir=LOG_DIR,
    grad_clip=0.5,
    conv_kernel=48,
    head_mode=HEAD_MODE,
)

model = WideBindStack(cfg).to(device)
n_params = model.param_count()
print(f'Model: {n_params:,} params ({n_params/1e6:.2f}M)')
print(f'Head:  {type(model.lm_head).__name__}')


In [ ]:
# @title 5. Auto Batch Size & Mixed Precision
def find_best_batch_size(model, seq_len, device, start=2):
    lo, hi = 1, start
    best = 1
    torch.cuda.empty_cache()
    try:
        x = torch.randint(0, 50000, (start, seq_len), device=device)
        h = model.embed_tokens(x)
        out, _, _ = model(h, None)
        out[:, :1].sum().backward()
        best = start
    except RuntimeError:
        hi = start // 2
    finally:
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
    while lo <= hi:
        mid = (lo + hi) // 2
        torch.cuda.empty_cache()
        try:
            x = torch.randint(0, 50000, (mid, seq_len), device=device)
            h = model.embed_tokens(x)
            out, _, _ = model(h, None)
            out[:, :1].sum().backward()
            best = mid
            lo = mid + 1
        except RuntimeError:
            hi = mid - 1
        finally:
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    torch.cuda.empty_cache()
    return max(1, best)

cfg.batch_size = find_best_batch_size(model, cfg.seq_len, device, start=2)
print(f'Optimal batch size: {cfg.batch_size}')
print(f'  Tokens/step: {cfg.batch_size * cfg.seq_len}')

scaler = torch.cuda.amp.GradScaler() if device == 'cuda' else None
if scaler:
    print('  Using fp16 mixed precision + GradScaler')

In [ ]:
# @title 6. Optimizer & LR Scheduler
param_groups = model.param_groups()
optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.95))

scheduler = MirrorLRScheduler(model, optimizer, cfg.lr,
    warmup=cfg.warmup_steps, target_var=cfg.target_var,
    mag_threshold=cfg.mag_threshold, lr_min_ratio=cfg.lr_min_ratio,
    max_decay_steps=cfg.max_decay_steps,
    var_min_for_lr_decay=cfg.var_min_for_lr_decay,
    cfg=cfg)
print('Scheduler: MirrorLRScheduler')

In [ ]:
# @title 7. Data Streams
class TokenStream:
    def __init__(self, path):
        self.data = np.memmap(path, dtype=np.uint16, mode='r')
        self.len = len(self.data)
    def get_batch(self, seq_len, batch_size, offset):
        needed = batch_size * seq_len + 1
        if offset + needed > self.len:
            offset = 0
        chunk = self.data[offset:offset + needed]
        x = torch.from_numpy(chunk[:batch_size * seq_len].reshape(batch_size, seq_len).copy())
        y = torch.from_numpy(chunk[1:batch_size * seq_len + 1].reshape(batch_size, seq_len).copy())
        return x.long(), y.long(), offset + batch_size * seq_len

stream_files = sorted(glob.glob(os.path.join(DATA_DIR, 'token_stream_*_clean.bin')))
if not stream_files:
    stream_files = sorted(glob.glob(os.path.join(DATA_DIR, 'token_stream_*.bin')))
if not stream_files:
    print('WARNING: No token_stream_*.bin found!')
    print(f'  Looked in: {DATA_DIR}')
    print('  Using random data for testing')
    streams = []
else:
    streams = [TokenStream(f) for f in stream_files]
    total_tokens = sum(s.len for s in streams)
    print(f'Found {len(streams)} files, {total_tokens:,} total tokens')

In [ ]:
# @title 8. Resume Checkpoint (optional)
start_step = 0
# Free live tensors left by a previous (possibly crashed) run — Colab keeps
# 'out'/'state'/'loss'/old references alive across cell re-runs, pinning GPU.
import gc as _gc
for _v in ('out', 'state', 'loss', 'ce_loss', 'aux_dict', 'grads', 'ot', 'opt'):
    try:
        globals().pop(_v, None)
    except Exception:
        pass
_gc.collect()
torch.cuda.empty_cache()
print(f'  Free after cleanup: {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB')

state = None
best_val_loss = float('inf')

ckpt_files = sorted(
    glob.glob(os.path.join(SAVE_DIR, 'step_*.pt')),
    key=lambda p: int(os.path.basename(p).split('_')[1].split('.')[0]))

if not ckpt_files:
    # Also check for best.pt
    best_ckpt = os.path.join(SAVE_DIR, 'best.pt')
    if os.path.exists(best_ckpt):
        ckpt_files = [best_ckpt]

if ckpt_files:
    latest = ckpt_files[-1]
    print(f'Resuming from {latest}')
    ckpt = torch.load(latest, map_location=device, weights_only=True)

    # ── Consistency: vocab / arch / head ──
    ck_cfg = ckpt.get('cfg', None)
    arch_mismatch = False
    if ck_cfg is not None:
        print(f'  ckpt cfg:  vocab={getattr(ck_cfg, "vocab", "?")} '
              f'D={getattr(ck_cfg, "D", "?")} L={getattr(ck_cfg, "n_layers", "?")} '
              f'head_mode={getattr(ck_cfg, "head_mode", "partitioned")}')
        print(f'  model cfg: vocab={cfg.vocab} D={cfg.D} L={cfg.n_layers} '
              f'head_mode={getattr(cfg, "head_mode", "partitioned")}')
        arch_mismatch = (getattr(ck_cfg, 'vocab', None) != cfg.vocab or
                         getattr(ck_cfg, 'D', None) != cfg.D or
                         getattr(ck_cfg, 'n_layers', None) != cfg.n_layers)
        if arch_mismatch:
            print('  WARNING: ARCH/VOCAB MISMATCH — mismatched tensors will be SKIPPED '
                  '(model keeps its current init; training continues without crash).')
        if getattr(ck_cfg, 'head_mode', 'partitioned') != getattr(cfg, 'head_mode', 'partitioned'):
            print('  NOTE: head_mode differs from checkpoint — new head params '
                  '(bit_bias/log_temp/_prop) stay at init.')

    # ── Load weights by shape (strict=False CRASHES on shape mismatch,
    #    so load per-tensor and skip what does not match) ──
    sd = ckpt['model']
    model_sd = model.state_dict()
    filtered, skipped, missing, unexpected = {}, [], [], []
    for k, v in sd.items():
        if k not in model_sd:
            unexpected.append(k)
        elif v.shape != model_sd[k].shape:
            skipped.append(k)
        else:
            filtered[k] = v
    for k in model_sd:
        if k not in sd:
            missing.append(k)
    model.load_state_dict(filtered, strict=False)
    print(f'  Weights: loaded {len(filtered)}, skipped {len(skipped)} (shape), '
          f'missing {len(missing)}, unexpected {len(unexpected)}')

    # ── Optimizer: drop state entries whose shape no longer matches the model
    #    (vocab-dim params, head swap) instead of crashing at optimizer.step() ──
    try:
        optimizer.load_state_dict(ckpt['optimizer'])
        for group in optimizer.param_groups:
            for p in group['params']:
                st = optimizer.state.get(p)
                if st:
                    for k in list(st):
                        t = st[k]
                        if isinstance(t, torch.Tensor) and t.shape != p.shape:
                            del st[k]
    except Exception as e:
        print(f'  Optimizer state reset ({type(e).__name__}): {e}')
        optimizer = torch.optim.AdamW(model.param_groups(), betas=(0.9, 0.95))
        scheduler = MirrorLRScheduler(model, optimizer, cfg.lr,
            warmup=cfg.warmup_steps, target_var=cfg.target_var,
            mag_threshold=cfg.mag_threshold, lr_min_ratio=cfg.lr_min_ratio,
            max_decay_steps=cfg.max_decay_steps,
            var_min_for_lr_decay=cfg.var_min_for_lr_decay,
            cfg=cfg)

    if 'scheduler' in ckpt:
        try:
            scheduler.load_state_dict(ckpt['scheduler'])
        except Exception as e:
            print(f'  Scheduler state reset ({type(e).__name__})')
    start_step = ckpt.get('step', 0)
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    if skipped or arch_mismatch:
        best_val_loss = float('inf')  # old metric/weights not comparable
        print('  Reset best_val_loss=inf (weights/metric not comparable to checkpoint)')
    print(f'  Resumed at step {start_step}')
else:
    print('No checkpoint found, starting fresh')


In [ ]:
# @title 9. 🚀 TRAINING LOOP
import torch._dynamo
torch._dynamo.config.suppress_errors = True

stream_idx = 0
offset = 0
tokens_seen = 0
t0 = time.time()
rng = torch.Generator(device='cpu').manual_seed(42)

print(f'Training: step {start_step} -> {cfg.max_steps}')
print(f'  ({cfg.max_steps - start_step} steps remaining)\n')

try:
    for step in range(start_step, cfg.max_steps):
        model.train()

        # ── Data ──
        if streams:
            if offset == 0:
                stream_idx = torch.randint(0, len(streams), (1,), generator=rng).item()
                state = None
            x, y, offset = streams[stream_idx].get_batch(cfg.seq_len, cfg.batch_size, offset)
        else:
            # Fallback: random data
            x = torch.randint(0, cfg.vocab, (cfg.batch_size, cfg.seq_len))
            y = torch.randint(0, cfg.vocab, (cfg.batch_size, cfg.seq_len))

        x, y = x.to(device), y.to(device)

        # ── Forward ──
        with torch.amp.autocast(device_type='cuda', enabled=scaler is not None):
            h = model.embed_tokens(x)
            out, state, _ = model(h, state, step=step)
            ce_loss, aux_dict = model.compute_losses(out, y)
            loss = ce_loss + sum(aux_dict.values())

        # ── Backward ──
        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        tokens_seen += cfg.batch_size * cfg.seq_len

        # ── Optimizer step ──
        if scaler:
            if cfg.grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            if cfg.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        # ── Detach state ──
        if state is not None:
            state = tuple(tuple(t.detach() if isinstance(t, torch.Tensor) else t for t in s) if s else None for s in state)

        # ── Log ──
        if step % cfg.log_interval == 0:
            dt = time.time() - t0
            tok_s = tokens_seen / max(dt, 1e-8)
            lr = scheduler.get_last_lr()[0]
            mem_gb = torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else 0
            aux_str = ' '.join(f'{k}={v:.4f}' for k, v in sorted(aux_dict.items()) if abs(v) > 1e-6)
            print(f'step={step:>6}  loss={loss.item():.4f}  ce={ce_loss.item():.4f}  '
                  f'lr={lr:.2e}  tok/s={tok_s:.0f}  mem={mem_gb:.1f}GB')
            if aux_str:
                print(f'  aux: {aux_str}')
            if device == 'cuda':
                torch.cuda.reset_peak_memory_stats()

        # ── Eval ──
        if step > 0 and step % cfg.eval_interval == 0:
            model.eval()
            val_loss = 0.0
            n_val = 0
            with torch.no_grad():
                for s in streams[:3]:  # eval on first 3 streams
                    voff = max(s.len // 4, cfg.batch_size * cfg.seq_len + 1)
                    for _ in range(min(100, s.len // (cfg.batch_size * cfg.seq_len))):
                        vx, vy, voff = s.get_batch(cfg.seq_len, cfg.batch_size, voff)
                        if voff == 0:
                            break
                        vx, vy = vx.to(device), vy.to(device)
                        h = model.embed_tokens(vx)
                        out, _, _ = model(h, None, adaptive=False)
                        ce, _ = model.compute_losses(out, vy)
                        val_loss += ce.item()
                        n_val += 1
            if n_val > 0:
                val_loss /= n_val
                val_ppl = math.exp(min(val_loss, 20))
                print(f'  EVAL step={step}: val_loss={val_loss:.4f} val_ppl={val_ppl:.2f}')
                scheduler.report_val_loss(val_loss)
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save({
                        'step': step, 'model': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'scheduler': scheduler.state_dict(),
                        'best_val_loss': best_val_loss, 'cfg': cfg,
                    }, os.path.join(SAVE_DIR, 'best.pt'))
                    print(f'  Saved best to best.pt')
            model.train()
            if device == 'cuda':
                torch.cuda.empty_cache()

        # ── Save ──
        if step > 0 and step % cfg.save_interval == 0:
            save_path = os.path.join(SAVE_DIR, f'step_{step}.pt')
            torch.save({
                'step': step, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'best_val_loss': best_val_loss, 'cfg': cfg,
            }, save_path)
            print(f'  Saved checkpoint to {save_path}')

except KeyboardInterrupt:
    print('\nInterrupted — saving...')
    save_path = os.path.join(SAVE_DIR, f'interrupt_step_{step}.pt')
    torch.save({
        'step': step, 'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_val_loss': best_val_loss, 'cfg': cfg,
    }, save_path)
    print(f'Saved to {save_path}')

print('\nTraining complete!')